# Feres Analysis

Stroke territory cohort analysis (ACS vs PCS) on `stroke:1.0` using `mip.Context` / `mip.Analysis`.

This notebook reproduces a federated analysis workflow from the FERES (Federating European REgistries for Stroke) deployment of the Medical Informatics Platform.

## Reference

Mailli E, Michel P, Bastic V, Dhallenne J, Filippopolitis K, Katsouli M-O, Karampatsis T-M, Glenis A (2026). Deploying the Medical Informatics Platform for cross-border federated analytics in FERES and eCREAM. *Frontiers in Disaster and Emergency Medicine* 4:1748193. [https://doi.org/10.3389/femer.2026.1748193](https://www.frontiersin.org/journals/disaster-and-emergency-medicine/articles/10.3389/femer.2026.1748193/full)

In [ ]:
import mip
from mip import Analysis, Case, Context, FilterGroup, Rule, Validation, catalog, configure

# No manual token needed when running via /notebook (PLATFORM_TOKEN is injected).
# If running outside docker-compose, you likely need:
#   configure(base_url='http://localhost:8080/services', token='...')
configure()
print('Public exports:', ', '.join(mip.__all__))

In [ ]:
# Discover available data models and datasets
catalog.models().to_dataframe()

In [ ]:
catalog.datasets('stroke:1.0').to_dataframe()

In [ ]:
catalog.visualize('stroke:1.0', include_variables=True)

In [ ]:
context = Context(
    data_model='stroke:1.0',
    datasets=['ssrdataset_harmonized'],
    mip_version='dev',
)

analysis = Analysis(context)
analysis

In [ ]:
ACS_CODES = ['anterior_left', 'anterior_right']
PCS_CODES = ['posterior_left', 'posterior_right']

territory_cohort = analysis.transformations.categorical_from_filters(
    name='stroke_territory_cohort',
    label='Stroke territory cohort',
    cases=[
        Case(
            label='ACS',
            when=FilterGroup.and_(Rule('stroke_territory', 'in', ACS_CODES)),
        ),
        Case(
            label='PCS',
            when=FilterGroup.and_(Rule('stroke_territory', 'in', PCS_CODES)),
        ),
    ],
    otherwise=mip.MISSING,
    validation=Validation(mutually_exclusive=True, allow_unmatched=True),
)

analysis = analysis.with_transformations([territory_cohort])

In [ ]:
numeric_summary = analysis.describe.numeric(
    variables=['age', 'nihss_24h'],
    group_by='stroke_territory_cohort',
    levels=['ACS', 'PCS'],
)
numeric_summary.to_dataframe()

In [ ]:
ttest_results = analysis.tests.ttest_independent(
    variables=['age'],
    group_by='stroke_territory_cohort',
    group_a='ACS',
    group_b='PCS',
)
ttest_results.to_dataframe()